# Tutorial 03: Collective coupling — fixed $\lambda\sqrt{N}$, growing $N$

Tutorials 01–02 used a **single** A–A dimer. Here we grow the number of identical dimers

$$N \in \{1, 2, 4, 8\}$$

while keeping the **collective** light–matter strength fixed:

$$g \equiv \lambda\sqrt{N} = \text{constant} \quad\Rightarrow\quad \lambda(N) = \frac{g}{\sqrt{N}}.$$

As $N$ increases, the per-molecule coupling $\lambda$ shrinks, but the vacuum Rabi physics that enters the IR spectrum is controlled by $g$. With identical non-interacting (or weakly interacting) molecules, the IR lineshape should therefore be **the same for every $N$**.

What we will do:

1. Import the OpenMM cavity-MD stack (same as Tutorial 02)
2. Fix $g$ and build $\lambda(N)$ for each system size
3. Run short NVT trajectories for $N = 1,2,4,8$
4. Overlay the IR spectra and check that they match

**Kernel:** use the same pixi `test` environment kernel as Tutorials 01–02.

### Unit conventions

mKA parameters are in atomic units; OpenMM stores kJ/mol, nm, ps. Conversions go through `openmm.cavitymd.constants.Units`.


## 1. Import the codebase


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import openmm
from openmm import unit
from scipy import fftpack, signal

from openmm.cavitymd.constants import Units
from openmm.cavitymd.forcefields.mka import (
    MASS_A, PHOTON_MASS_AMU,
    K_AA_AU, R0_AA_AU,
    CHARGE_MAG, OMEGA_C_CM1,
)
from openmm.cavitymd.thermostats import DualThermostat

print("OpenMM version:", openmm.__version__)
for i in range(openmm.Platform.getNumPlatforms()):
    print(" platform:", openmm.Platform.getPlatform(i).getName())


Next we define helpers shared by every $N$: dipole magnitude, the IR post-processing recipe from Tutorials 01–02, and a builder that constructs $N$ A–A dimers + one photon with the Tutorial 02 wiring (`PHOTON_MASS_AMU`, Bussi on molecules only, displacer at step 0).


In [ ]:
def dipole_magnitude(pos, charges, indices):
    """Translation/rotation-invariant dipole magnitude (|Σ qᵢ rᵢ|)."""
    d = np.dot(charges, pos[indices])
    return np.linalg.norm(d)


def ir_spectrum_from_dipole(dipoles, dt_fs, T_K, fraction_acf=0.25):
    """Dipole ACF -> DCT lineshape x omega field factor (same recipe as Tutorials 01-02)."""
    dipole_signal = np.asarray(dipoles, dtype=float) - np.mean(dipoles)
    N_sig = len(dipole_signal)
    n_acf = max(3, int(N_sig * fraction_acf))
    if N_sig % 2 == 0:
        shifted = np.zeros(2 * N_sig)
    else:
        shifted = np.zeros(2 * N_sig - 1)
    shifted[N_sig // 2 : N_sig // 2 + N_sig] = dipole_signal
    acf_full = signal.fftconvolve(shifted, dipole_signal[::-1], mode="same")[-N_sig:] / np.arange(N_sig, 0, -1)
    autocorr = acf_full[:n_acf]

    timestep = dt_fs * 1e-15
    lineshape = fftpack.dct(autocorr, type=1)[1:]
    freqs_hz = np.linspace(0, 0.5 / timestep, len(autocorr))[1:]
    freqs_cm1 = freqs_hz / (100.0 * 299792458.0)

    boltz = 1.38064852e-23
    hbar = 1.05457180013e-34
    field = freqs_hz * (1.0 - np.exp(-hbar * freqs_hz / (boltz * T_K)))
    spectrum = lineshape * field
    return freqs_cm1, spectrum, autocorr


def build_n_aa_dimers_cavity(n_mol, lambda_coupling, omegac_au, sep_nm, T_K, seed, dt_ps):
    """N identical A-A dimers + photon; Tutorial-02-style forces / thermostat / displacer.

    Inter-molecular Coulomb is turned off (exceptions) so the sqrt(N) scaling is not
    contaminated by molecule-molecule electrostatics. Charges remain on the
    NonbondedForce so CavityForce still sees the total molecular dipole.
    """
    system = openmm.System()
    bond_force = openmm.HarmonicBondForce()
    nb = openmm.NonbondedForce()
    nb.setNonbondedMethod(openmm.NonbondedForce.NoCutoff)

    positions = []
    mol_indices = []
    charges = []

    # Place dimers on a rough cubic grid with spacing sep_nm
    side = int(np.ceil(n_mol ** (1.0 / 3.0)))
    mol_id = 0
    for ix in range(side):
        for iy in range(side):
            for iz in range(side):
                if mol_id >= n_mol:
                    break
                cx = (ix - 0.5 * (side - 1)) * sep_nm
                cy = (iy - 0.5 * (side - 1)) * sep_nm
                cz = (iz - 0.5 * (side - 1)) * sep_nm
                i0 = system.getNumParticles()
                system.addParticle(MASS_A)
                system.addParticle(MASS_A)
                bond_force.addBond(i0, i0 + 1, r0_aa_omm, k_aa_omm)
                nb.addParticle(-CHARGE_MAG, 0.1, 0.0)
                nb.addParticle(+CHARGE_MAG, 0.1, 0.0)
                nb.addException(i0, i0 + 1, 0.0, 1.0, 0.0)
                positions.append(openmm.Vec3(cx - half_r0, cy, cz) * unit.nanometer)
                positions.append(openmm.Vec3(cx + half_r0, cy, cz) * unit.nanometer)
                mol_indices.extend([i0, i0 + 1])
                charges.extend([-CHARGE_MAG, +CHARGE_MAG])
                mol_id += 1
            if mol_id >= n_mol:
                break
        if mol_id >= n_mol:
            break

    # Zero intermolecular Coulomb so spectra isolate collective cavity coupling
    for m1 in range(n_mol):
        for m2 in range(m1 + 1, n_mol):
            for a in (0, 1):
                for b in (0, 1):
                    i = 2 * m1 + a
                    j = 2 * m2 + b
                    nb.addException(i, j, 0.0, 1.0, 0.0)

    photon_index = system.addParticle(PHOTON_MASS_AMU)
    nb.addParticle(0.0, 0.1, 0.0)
    positions.append(openmm.Vec3(0, 0, 0) * unit.nanometer)

    system.addForce(bond_force)
    system.addForce(nb)

    cavity_force = openmm.CavityForce(photon_index, omegac_au, lambda_coupling, PHOTON_MASS_AMU)
    system.addForce(cavity_force)

    displacer = openmm.CavityParticleDisplacer(photon_index, omegac_au, PHOTON_MASS_AMU)
    displacer.setSwitchOnStep(0)
    displacer.setSwitchOnLambda(lambda_coupling)
    system.addForce(displacer)

    DualThermostat.setup_bussi_for_system(
        system,
        molecular_indices=mol_indices,
        temperature_K=T_K,
        tau_ps=1.0,
        random_number_seed=seed,
    )

    integrator = openmm.VerletIntegrator(dt_ps)
    platform = openmm.Platform.getPlatformByName("CUDA")
    platform.setPropertyDefaultValue("Precision", "mixed")
    context = openmm.Context(system, integrator, platform)
    context.setPositions(positions)
    context.setVelocitiesToTemperature(T_K, seed)

    # Fire finite-q displacer at step 0
    integrator.step(1)
    return system, context, integrator, mol_indices, np.asarray(charges, dtype=float), photon_index


## 2. Set simulation parameters

We fix the collective coupling $g = \lambda\sqrt{N}$ and derive $\lambda(N)$. Simulation parameters otherwise match Tutorial 02.


In [ ]:
# ============================================================
# Tunable parameters: change these to explore the physics
# ============================================================
g_collective = 0.5      # fixed lambda*sqrt(N) (a.u.); try 0.0, 0.5, 1.0
N_list       = [1, 2, 4, 8]
T_K          = 100.0
dt_fs        = 1.0
N_steps      = 12_000   # per-N production steps (~12 ps); increase for smoother spectra
seed         = 42
sep_nm       = 2.0      # grid spacing between dimer centers

# Derived constants
dt_ps     = dt_fs * 1e-3
omegac_au = Units.cm1_to_au(OMEGA_C_CM1)
B2NM      = Units.BOHR_TO_NM
H2K       = Units.HARTREE_TO_KJMOL
k_aa_omm  = K_AA_AU * H2K / B2NM**2
r0_aa_omm = R0_AA_AU * B2NM
half_r0   = r0_aa_omm / 2.0

print(f"g = lambda*sqrt(N) = {g_collective}")
print(f"omega_c = {OMEGA_C_CM1} cm^-1  ({omegac_au:.6e} Hartree)")
print(f"T = {T_K} K,  dt = {dt_fs} fs,  steps/N = {N_steps}")
print("N   lambda = g/sqrt(N)   lambda*sqrt(N)")
for N in N_list:
    lam = g_collective / np.sqrt(N)
    print(f"{N:2d}  {lam:.6f}  {lam * np.sqrt(N):.6f}")


---
## 3. Theory: why $\lambda\sqrt{N}$ is the right invariant

### Collective vacuum Rabi physics

For $N$ identical molecules coupled to one cavity mode, the bright-state coupling scales as

$$\Omega_R \propto \lambda\sqrt{N} \equiv g.$$

Dark states (combinations of molecular dipoles orthogonal to the bright mode) do not mix with the photon at linear order. The IR spectrum of the **total** dipole therefore depends on $g$, not on $\lambda$ and $N$ separately.

### Protocol in this notebook

| $N$ | $\lambda = g/\sqrt{N}$ | $g=\lambda\sqrt{N}$ |
|---:|---:|---:|
| 1 | $g$ | $g$ |
| 2 | $g/\sqrt{2}$ | $g$ |
| 4 | $g/2$ | $g$ |
| 8 | $g/\sqrt{8}$ | $g$ |

### Finite-$q$ displacement

Same as Tutorial 02: `CavityParticleDisplacer` with `setSwitchOnStep(0)`, `setSwitchOnLambda(λ)`, and matching `PHOTON_MASS_AMU`.

### Inter-molecular forces

We **zero intermolecular Coulomb** (OpenMM exceptions) so the overlay tests pure collective cavity scaling. Each dimer still has its A–A harmonic bond; the cavity sees the full set of partial charges through `NonbondedForce`.


## 4. Run $N = 1,2,4,8$ and collect IR spectra

Each system uses Tutorial 02's NVT stack. We record the **total** molecular dipole magnitude $|\mathbf{d}_\mathrm{tot}(t)| = \big|\sum_{i\in\mathrm{mol}} q_i\mathbf{r}_i\big|$ and convert it to an IR spectrum.


In [ ]:
kB_kJ = Units.KB_KJMOL_PER_K
spectra = {}  # N -> dict of results

for N in N_list:
    lam = g_collective / np.sqrt(N)
    system, context, integrator, mol_indices, charges, photon_index = build_n_aa_dimers_cavity(
        n_mol=N,
        lambda_coupling=lam,
        omegac_au=omegac_au,
        sep_nm=sep_nm,
        T_K=T_K,
        seed=seed + N,   # different seed per N; same g
        dt_ps=dt_ps,
    )

    pos0 = context.getState(getPositions=True).getPositions(asNumpy=True).value_in_unit(unit.nanometer)
    print(f"\n=== N = {N}, lambda = {lam:.6f}, lambda*sqrt(N) = {lam * np.sqrt(N):.6f} ===")
    print(f" particles = {system.getNumParticles()}, photon index = {photon_index}")
    print(
        f" photon q after displacer = "
        f"({pos0[photon_index,0]:.4f}, {pos0[photon_index,1]:.4f}, {pos0[photon_index,2]:.4f}) nm"
    )

    dof = 3 * len(mol_indices) - 3  # Bussi subtracts CM motion
    dipoles = []
    T_kin = []
    for _ in range(N_steps):
        integrator.step(1)
        state = context.getState(getPositions=True, getVelocities=True)
        pos = state.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
        vel = state.getVelocities(asNumpy=True).value_in_unit(unit.nanometer / unit.picosecond)
        dipoles.append(dipole_magnitude(pos, charges, mol_indices))
        ke_mol = 0.0
        for i in mol_indices:
            m = system.getParticleMass(i).value_in_unit(unit.dalton)
            ke_mol += 0.5 * m * float(np.dot(vel[i], vel[i]))
        T_kin.append(2.0 * ke_mol / (dof * kB_kJ))

    dipoles = np.asarray(dipoles)
    T_kin = np.asarray(T_kin)
    freqs, spectrum, _ = ir_spectrum_from_dipole(dipoles, dt_fs, T_K)
    spectra[N] = {
        "lambda": lam,
        "freqs_cm1": freqs,
        "spectrum": spectrum,
        "mean_T": float(T_kin.mean()),
        "dipoles": dipoles,
    }
    print(f" mean T_kin = {T_kin.mean():.1f} K  (target {T_K} K)")

print("\nDone.")


## 5. Overlay IR spectra

Normalize each spectrum to its own maximum and plot them together. If $g=\lambda\sqrt{N}$ is what controls the polariton physics, the curves for different $N$ should lie on top of each other (up to sampling noise).


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for N in N_list:
    freqs = spectra[N]["freqs_cm1"]
    spec = spectra[N]["spectrum"]
    lam = spectra[N]["lambda"]
    mask = (freqs > 0) & (freqs <= 4000)
    s = spec[mask]
    if s.max() > 0:
        s = s / s.max()
    ax.plot(
        freqs[mask], s, lw=1.0,
        label=f"N={N}, lambda={lam:.4f}, g={lam * np.sqrt(N):.3f}",
    )

ax.axvline(OMEGA_C_CM1, color="k", ls="--", lw=1.0, label=f"omega_c = {OMEGA_C_CM1:.0f} cm^-1")
ax.set_xlim(0, 4000)
ax.set_xlabel("Frequency (cm^-1)")
ax.set_ylabel("Intensity (arb., peak-normalized)")
ax.set_title(f"IR overlay at fixed g = lambda*sqrt(N) = {g_collective}")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Zoom on the vibrational / polariton window
fig, ax = plt.subplots(figsize=(9, 4))
for N in N_list:
    freqs = spectra[N]["freqs_cm1"]
    spec = spectra[N]["spectrum"]
    mask = (freqs > 1000) & (freqs < 2200)
    s = spec[mask]
    if s.max() > 0:
        s = s / s.max()
    ax.plot(freqs[mask], s, lw=1.1, label=f"N={N}")
ax.axvline(OMEGA_C_CM1, color="k", ls="--", lw=1.0)
ax.set_xlabel("Frequency (cm^-1)")
ax.set_ylabel("Intensity (arb., peak-normalized)")
ax.set_title("Zoom: 1000-2200 cm^-1")
ax.legend()
plt.tight_layout()
plt.show()

print("Mean molecular temperatures:")
for N in N_list:
    print(f"  N={N}: T_kin = {spectra[N]['mean_T']:.1f} K")


### Optional check: break the invariant

Re-run with **fixed** $\lambda$ (no $1/\sqrt{N}$ rescaling). The spectra should then **split apart** as $N$ grows, because $g=\lambda\sqrt{N}$ increases. Set `RUN_FIXED_LAMBDA = True` below if you want that contrast.


In [ ]:
# Contrast experiment: fixed lambda, growing N  ->  g = lambda*sqrt(N) increases
RUN_FIXED_LAMBDA = False
lambda_fixed = g_collective  # same as N=1 value

if RUN_FIXED_LAMBDA:
    spectra_fixed = {}
    for N in N_list:
        system, context, integrator, mol_indices, charges, photon_index = build_n_aa_dimers_cavity(
            n_mol=N, lambda_coupling=lambda_fixed, omegac_au=omegac_au,
            sep_nm=sep_nm, T_K=T_K, seed=seed + 100 + N, dt_ps=dt_ps,
        )
        dipoles = []
        for _ in range(N_steps):
            integrator.step(1)
            pos = context.getState(getPositions=True).getPositions(asNumpy=True).value_in_unit(unit.nanometer)
            dipoles.append(dipole_magnitude(pos, charges, mol_indices))
        freqs, spectrum, _ = ir_spectrum_from_dipole(dipoles, dt_fs, T_K)
        spectra_fixed[N] = (freqs, spectrum, lambda_fixed * np.sqrt(N))
        print(f"fixed-lambda N={N}: g=lambda*sqrt(N)={lambda_fixed * np.sqrt(N):.3f}")

    fig, ax = plt.subplots(figsize=(9, 4))
    for N in N_list:
        freqs, spec, gN = spectra_fixed[N]
        mask = (freqs > 1000) & (freqs < 2200)
        s = spec[mask]
        if s.max() > 0:
            s = s / s.max()
        ax.plot(freqs[mask], s, lw=1.1, label=f"N={N}, g={gN:.3f}")
    ax.axvline(OMEGA_C_CM1, color="k", ls="--", lw=1.0)
    ax.set_title(f"Contrast: fixed lambda={lambda_fixed} (g grows with sqrt(N))")
    ax.set_xlabel("Frequency (cm^-1)")
    ax.set_ylabel("Intensity (arb.)")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Skipped fixed-lambda contrast (set RUN_FIXED_LAMBDA = True to enable).")


### Expected results

- For fixed $g=\lambda\sqrt{N}$, peak-normalized IR spectra for $N=1,2,4,8$ should **overlap** (noise decreases if you raise `N_steps`).
- Mean molecular $T_\mathrm{kin}$ should sit near the Bussi bath temperature for every $N$.
- With `RUN_FIXED_LAMBDA = True`, spectra **separate** as $g$ grows with $\sqrt{N}$.

### Tutorial series summary

| Tutorial | Ensemble | System | Key lesson |
|---|---|---|---|
| 01 | NVE | 1× A–A dimer | coherent oscillations, force wiring |
| 02 | NVT | 1× A–A dimer | canonical sampling, photon not in bath |
| 03 | NVT | $N=1,2,4,8$ A–A dimers | collective coupling: IR invariant when $\lambda\sqrt{N}$ fixed |


---
## Reference: mKA constants

| Quantity | Symbol | Value |
|----------|--------|-------|
| Cavity frequency | ω_c | 1560 cm⁻¹ |
| A–A bond length | r₀ | 2.282 Bohr = 0.121 nm |
| A–A bond stiffness | K_AA | 0.732 Ha/Bohr² |
| Charge magnitude | q | ±0.3 e |
| A-atom mass | m_A | 16.0 amu |
| Photon mass | m_ph | 1/1822.9 amu |
